# Xavier (Glorot) and He initialisation
Why a spread of 1 / fan_in keeps the signal the same size from layer to layer, the four formulas,
and how to use them in Keras.

In [1]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"      # hide TensorFlow's start-up messages
import numpy as np
import pandas as pd
import keras
import tensorflow as tf
from sklearn.datasets import make_moons

tf.config.experimental.enable_op_determinism()   # same numbers on every run
np.set_printoptions(precision=4, suppress=True)

## The signal through 10 layers of 500 nodes (numpy, no training)
1000 rows of 500 standard-normal inputs. The weights of every layer are standard normal numbers times
a scale; biases 0. We record the standard deviation of each layer's activations.

In [2]:
acts = {"tanh": np.tanh, "relu": lambda v: np.maximum(0, v)}
n = 500
scales = {"0.01 (too small)": 0.01, "1 (too large)": 1.0,
          "Xavier: sqrt(1/fan_in)": np.sqrt(1 / n), "He: sqrt(2/fan_in)": np.sqrt(2 / n)}
rows = []
for act in ("tanh", "relu"):
    for name, s in scales.items():
        rng = np.random.default_rng(0)
        a = rng.standard_normal((1000, n))
        for layer in range(1, 11):
            a = acts[act](a @ (rng.standard_normal((n, n)) * s))
            rows.append(dict(activation=act, start=name, layer=layer, std=a.std()))
layer_std = pd.DataFrame(rows)
layer_std.to_csv("data/layer_std.csv", index=False)
print(layer_std[layer_std.layer.isin([1, 5, 10])].pivot_table(index=["activation", "start"], columns="layer",
                                                                values="std").round(4))

layer                                   1            5             10
activation start                                                     
relu       0.01 (too small)         0.1308       0.0001  0.000000e+00
           1 (too large)           13.0807  759405.5691  7.101535e+11
           He: sqrt(2/fan_in)       0.8273       0.7685  7.272000e-01
           Xavier: sqrt(1/fan_in)   0.5850       0.1358  2.270000e-02
tanh       0.01 (too small)         0.2138       0.0005  0.000000e+00
           1 (too large)            0.9821       0.9816  9.816000e-01
           He: sqrt(2/fan_in)       0.7216       0.5661  5.553000e-01
           Xavier: sqrt(1/fan_in)   0.6284       0.3203  2.266000e-01


## Why 1 / fan_in: the variance of a weighted sum
z = sum of n products w_i x_i. With independent w and x of mean 0, Var(z) = n Var(w) Var(x).

In [3]:
rng = np.random.default_rng(1)
x = rng.standard_normal((100_000, n))[:2000]         # 2000 rows, Var(x) = 1
for var_w in (0.01 ** 2, 1.0, 1 / n):
    w = rng.standard_normal(n) * np.sqrt(var_w)
    z = x @ w
    print(f"Var(w) = {var_w:.6f}: predicted Var(z) = n Var(w) = {n * var_w:.3f}, measured {z.var():.3f}")

Var(w) = 0.000100: predicted Var(z) = n Var(w) = 0.050, measured 0.053
Var(w) = 1.000000: predicted Var(z) = n Var(w) = 500.000, measured 456.725
Var(w) = 0.002000: predicted Var(z) = n Var(w) = 1.000, measured 0.932


## Keras' initialisers: check their spread
fan_in = 10, fan_out = 10 (a 10 x 10 layer); 100 000 numbers each.

In [4]:
fan_in = fan_out = 10
for name in ("glorot_normal", "glorot_uniform", "he_normal", "he_uniform"):
    w = keras.initializers.get(name)
    w = w.__class__(seed=0)((1000, 100)).numpy()          # 1000 inputs x 100 outputs
    fi, fo = 1000, 100
    target = {"glorot_normal": np.sqrt(2 / (fi + fo)), "he_normal": np.sqrt(2 / fi),
              "glorot_uniform": np.sqrt(6 / (fi + fo)), "he_uniform": np.sqrt(6 / fi)}[name]
    kind = "limit" if "uniform" in name else "std"
    print(f"{name:15s} measured std {w.std():.4f}, largest |w| {np.abs(w).max():.4f}, formula {kind} {target:.4f}")

glorot_normal   measured std 0.0426, largest |w| 0.0969, formula std 0.0426
glorot_uniform  measured std 0.0427, largest |w| 0.0739, formula limit 0.0739
he_normal       measured std 0.0447, largest |w| 0.1017, formula std 0.0447
he_uniform      measured std 0.0448, largest |w| 0.0775, formula limit 0.0775


## The moons network from the weight initialisation Note, with good starts
300 points, standardised; four hidden layers of 10 nodes; plain SGD, learning rate 0.1, 100 epochs.
First by hand: weights = randn * sqrt(1 / fan_in), biases 0, tanh.

In [5]:
X, y = make_moons(n_samples=300, noise=0.1, random_state=42)
X = ((X - X.mean(0)) / X.std(0)).astype("float32")

def network(act, init="glorot_uniform", seed=0):
    keras.utils.set_random_seed(seed)
    m = keras.Sequential([keras.Input(shape=(2,))]
                         + [keras.layers.Dense(10, activation=act, kernel_initializer=init) for _ in range(4)]
                         + [keras.layers.Dense(1, activation="sigmoid")])
    m.compile(optimizer=keras.optimizers.SGD(0.1), loss="binary_crossentropy", metrics=["accuracy"])
    return m

m_xavier = network("tanh")
rng = np.random.default_rng(1)
new = []
for w in m_xavier.get_weights():
    if w.ndim == 2:
        fan_in = w.shape[0]                             # rows = inputs coming into each node
        new.append((rng.standard_normal(w.shape) * np.sqrt(1 / fan_in)).astype("float32"))
    else:
        new.append(np.zeros_like(w))
m_xavier.set_weights(new)
print("first-layer weights (fan_in = 2):\n", m_xavier.get_weights()[0])
w0 = m_xavier.get_weights()[0].copy()
h = m_xavier.fit(X, y, epochs=100, batch_size=32, verbose=0).history
print(f"tanh, Xavier by hand: loss {h['loss'][0]:.3f} -> {h['loss'][-1]:.4f}, accuracy {h['accuracy'][-1]:.2f}")
print("first-layer weights after training:\n", m_xavier.get_weights()[0])
hist = {"tanh, Xavier by hand": h["loss"]}

first-layer weights (fan_in = 2):
 [[ 0.2444  0.581   0.2337 -0.9215  0.6402  0.3156 -0.3797  0.4109  0.2578
   0.208 ]
 [ 0.0201  0.3866 -0.5208 -0.1152 -0.3409  0.4234  0.0281 -0.2068 -0.5529
  -0.1819]]


tanh, Xavier by hand: loss 0.650 -> 0.0023, accuracy 1.00
first-layer weights after training:
 [[-0.1863  2.0367  0.4271 -1.7926  0.4591  0.226  -0.5498  0.4618  0.3705
   0.5069]
 [ 0.187   0.1599 -0.2269  0.1442 -0.2176  0.291   0.0301 -0.1797 -0.332
  -0.1506]]


The same with Keras' built-in initialisers, set per layer with `kernel_initializer`.

In [6]:
fits = {}
for act, init in (("relu", "he_normal"), ("relu", "he_uniform"), ("tanh", "glorot_normal"),
                  ("tanh", "glorot_uniform")):
    m = network(act, init)
    h = m.fit(X, y, epochs=100, batch_size=32, verbose=0).history
    fits[(act, init)] = m
    hist[f"{act}, {init}"] = h["loss"]
    print(f"{act:4s} + {init:14s}: loss {h['loss'][0]:.3f} -> {h['loss'][-1]:.4f}, accuracy {h['accuracy'][-1]:.2f}")
pd.DataFrame(hist).rename_axis("epoch").to_csv("data/loss_histories.csv")

relu + he_normal     : loss 0.770 -> 0.0020, accuracy 1.00


relu + he_uniform    : loss 0.590 -> 0.0012, accuracy 1.00


tanh + glorot_normal : loss 0.714 -> 0.0032, accuracy 1.00


tanh + glorot_uniform: loss 0.564 -> 0.0021, accuracy 1.00


Keras' default when `kernel_initializer` is not given:

In [7]:
print(keras.layers.Dense(10).kernel_initializer.__class__.__name__)
print(keras.layers.Dense(10).bias_initializer.__class__.__name__)

GlorotUniform
Zeros


Decision regions of the hand-made Xavier/tanh network and the He/ReLU network (for the figure).

In [8]:
g = np.linspace(-2.5, 2.5, 151)
gx, gy = np.meshgrid(g, g)
grid = np.c_[gx.ravel(), gy.ravel()].astype("float32")
pd.DataFrame({"x1": grid[:, 0], "x2": grid[:, 1],
              "xavier_tanh": m_xavier.predict(grid, verbose=0).ravel(),
              "he_relu": fits[("relu", "he_normal")].predict(grid, verbose=0).ravel()}).to_csv(
    "data/boundary_grid.csv", index=False, float_format="%.4g")
pd.DataFrame({"x1": X[:, 0], "x2": X[:, 1], "y": y}).to_csv("data/moons.csv", index=False)